In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("="*80)
print("📊 PREDICCIÓN DE APROBACIÓN - ALGEBRA LINEAL")
print("🤖 REGRESIÓN LOGÍSTICA")
print("="*80)

# =============================================================================
# 1. CARGAR DATOS
# =============================================================================
print("\n📂 Cargando datos concatenados...")
try:
    df =pd.read_csv(r'datos_concatenados.csv')

    print(f"✅ Datos cargados: {len(df):,} registros")
except FileNotFoundError:
    print("❌ No se encontró el archivo 'datosConcatenados.csv'")
    exit()

📊 PREDICCIÓN DE APROBACIÓN - ALGEBRA LINEAL
🤖 REGRESIÓN LOGÍSTICA

📂 Cargando datos concatenados...
✅ Datos cargados: 72,740 registros


In [29]:
# =============================================================================
# 2. FILTRAR
# =============================================================================
print("\n🔍 Filtrando Algebra Lineal...")
mask = df['Asignatura'].str.upper().str.contains('ALGEBRA LINEAL', na=False)
df_algebra_lineal = df[mask].copy()
print(f"✅ Registros de Algebra Lineal: {len(df_algebra_lineal):,}")
print(f"   Estudiantes únicos: {df_algebra_lineal['ALUMNO_ID'].nunique():,}")



🔍 Filtrando Algebra Lineal...
✅ Registros de Algebra Lineal: 1,980
   Estudiantes únicos: 1,664


In [30]:
print("\n🔍 Filtrando Algebra Moderna...")
mask = df['Asignatura'].str.upper().str.contains('ALGEBRA MODERNA', na=False)
df_algebra_moderna = df[mask].copy()
print(f"✅ Registros de Algebra Moderna: {len(df_algebra_moderna):,}")
print(f"   Estudiantes únicos: {df_algebra_moderna['ALUMNO_ID'].nunique():,}")


🔍 Filtrando Algebra Moderna...
✅ Registros de Algebra Moderna: 1,131
   Estudiantes únicos: 899


In [31]:
print("\n🔍 Filtrando Geometria Vectorial...")
mask = df['Asignatura'].str.upper().str.contains('GEOMETRIA VECTORIAL', na=False)
df_geometria_vectorial = df[mask].copy()
print(f"✅ Registros de Geometria Vectorial: {len(df_geometria_vectorial):,}")
print(f"   Estudiantes únicos: {df_geometria_vectorial['ALUMNO_ID'].nunique():,}")


🔍 Filtrando Geometria Vectorial...
✅ Registros de Geometria Vectorial: 1,187
   Estudiantes únicos: 935


In [32]:
mask_firma_AL=df_algebra_lineal['Firma']>50

In [33]:
df_anho_semestre = df_algebra_lineal.groupby(['Anho', 'Semestre']).agg({'ALUMNO_ID': 'nunique'}).reset_index()

In [34]:
df_anho_semestre.head(10)

,Anho,Semestre,ALUMNO_ID
0,2025,1,1220
1,2025,2,314
2,2026,1,444


In [50]:
np.sum(mask_firma_AL)

np.int64(526)

In [37]:
mask_firma_AL.value_counts()

,count
Firma,
False,1454
True,526


In [43]:
merge_aux=df_algebra_lineal.merge(df_algebra_moderna, on='ALUMNO_ID', how='inner')
nuevo=merge_aux.merge(df_geometria_vectorial, on='ALUMNO_ID', how='inner')

In [44]:
nuevo.head(10)

,ALUMNO_ID,Cod.Asign_x,Asignatura_x,Cod.Car.Sec_x,Cod.Curso_x,Convocatoria_x,Anho_x,Semestre_x,Doc.Firma_x,Aprobado_x,Anho.Firma_x,Primer.Par_x,Segundo.Par_x,Tercer.Par_x,TPLab._x,Lab._x,Proy._x,Pond.PP_x,Pond.SP_x,Pond.TPLab_x,Pond.Lab_x,Pond.Proy_x,Asis_x,Requisito_x,Firma_x,Primer.Rec_x,Segundo.Rec_x,Nota.Final_x,Archivo_Origen_x,FirmaCalculada_x,Carrera_x,Cod.Asign_y,Asignatura_y,Cod.Car.Sec_y,Cod.Curso_y,Convocatoria_y,Anho_y,Semestre_y,Doc.Firma_y,Aprobado_y,Anho.Firma_y,Primer.Par_y,Segundo.Par_y,Tercer.Par_y,TPLab._y,Lab._y,Proy._y,Pond.PP_y,Pond.SP_y,Pond.TPLab_y,Pond.Lab_y,Pond.Proy_y,Asis_y,Requisito_y,Firma_y,Primer.Rec_y,Segundo.Rec_y,Nota.Final_y,Archivo_Origen_y,FirmaCalculada_y,Carrera_y,Cod.Asign,Asignatura,Cod.Car.Sec,Cod.Curso,Convocatoria,Anho,Semestre,Doc.Firma,Aprobado,Anho.Firma,Primer.Par,Segundo.Par,Tercer.Par,TPLab.,Lab.,Proy.,Pond.PP,Pond.SP,Pond.TPLab,Pond.Lab,Pond.Proy,Asis,Requisito,Firma,Primer.Rec,Segundo.Rec,Nota.Final,Archivo_Origen,FirmaCalculada,Carrera
0,FIUNA_ALUMNO_006371,23023,ALGEBRA LINEAL,MCT-PLS23,2,1,2025,1,0,N,0,0,0,0,15,0,10,30,40,20,0,0,0,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,MCT,23014,ALGEBRA MODERNA,MCT-PLS23,1,1,2024,2,37163,S,2024,65,40,0,0,0,0,50,50,0,0,0,2,1,53.00,0,0,2F-2,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,MCT,23013,GEOMETRIA VECTORIAL,MCT-PLS23,1,1,2024,2,37578,S,2024,63,35,0,60,0,0,45,45,10,0,0,1,1,50.00,0,0,2F-2,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,MCT
1,FIUNA_ALUMNO_006797,23023,ALGEBRA LINEAL,MCT-PLS23,2,1,2025,1,0,N,0,0,0,0,15,0,10,30,40,20,0,0,0,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,MCT,23014,ALGEBRA MODERNA,MCT-PLS23,1,1,2024,2,37163,S,2024,65,40,0,0,0,0,50,50,0,0,0,2,1,53.00,0,0,2F-2,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,MCT,23013,GEOMETRIA VECTORIAL,MCT-PLS23,1,1,2024,2,37578,S,2024,63,35,0,60,0,0,45,45,10,0,0,1,1,50.00,0,0,2F-2,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,MCT
2,FIUNA_ALUMNO_011263,23023,ALGEBRA LINEAL,ELE-PLS23,2,1,2025,1,0,N,0,6,0,0,5,0,10,30,40,20,0,0,2,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,ELE,23014,ALGEBRA MODERNA,ELE-PLS23,1,1,2024,2,37163,S,2024,60,68,0,0,0,0,50,50,0,0,0,1,1,64.00,0,0,"1F-1,2F-2",rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE,23013,GEOMETRIA VECTORIAL,ELE-PLS23,1,1,2024,2,37578,N,2024,60,36,0,90,0,0,45,45,10,0,0,1,1,52.00,0,0,2F-1,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE
3,FIUNA_ALUMNO_011263,23023,ALGEBRA LINEAL,ELE-PLS23,2,1,2025,1,0,N,0,6,0,0,5,0,10,30,40,20,0,0,2,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,ELE,23014,ALGEBRA MODERNA,ELE-PLS23,1,1,2024,2,37163,S,2024,60,68,0,0,0,0,50,50,0,0,0,1,1,64.00,0,0,"1F-1,2F-2",rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE,23013,GEOMETRIA VECTORIAL,ELE-PLS23,1,1,2025,2,37578,N,2024,0,0,0,90,0,0,45,45,10,0,0,1,1,9.00,0,0,NaN,rendimiento_año_2025_ciclo_2_anon.xlsx,NaN,ELE
4,FIUNA_ALUMNO_012732,23023,ALGEBRA LINEAL,ELE-PLS23,2,1,2025,1,0,N,0,6,0,0,5,0,10,30,40,20,0,0,2,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,ELE,23014,ALGEBRA MODERNA,ELE-PLS23,1,1,2024,2,37163,S,2024,60,68,0,0,0,0,50,50,0,0,0,1,1,64.00,0,0,"1F-1,2F-2",rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE,23013,GEOMETRIA VECTORIAL,ELE-PLS23,1,1,2024,2,37578,N,2024,60,36,0,90,0,0,45,45,10,0,0,1,1,52.00,0,0,2F-1,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE
5,FIUNA_ALUMNO_012732,23023,ALGEBRA LINEAL,ELE-PLS23,2,1,2025,1,0,N,0,6,0,0,5,0,10,30,40,20,0,0,2,0,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,3.00,ELE,23014,ALGEBRA MODERNA,ELE-PLS23,1,1,2024,2,37163,S,2024,60,68,0,0,0,0,50,50,0,0,0,1,1,64.00,0,0,"1F-1,2F-2",rendimiento_año_2024_ciclo_2_anon.xlsx,NaN,ELE,23013,GEOMETRIA VECTORIAL,ELE-PLS23,1,1,2025,2,37578,N,2024,0,0,0,90,0,0,45,45,10,0,0,1,1,9.00,0,0,NaN,rendimiento_año_2025_ciclo_2_anon.xlsx,NaN,ELE
6,FIUNA_ALUMNO_010887,23023,ALGEBRA LINEAL,CIV-PLS23,2,1,2025,1,0,N,0,40,0,0,43,0,10,30,40,20,0,0,1,1,0.00,0,0,NaN,rendimiento_año_2025_ciclo_1_anon.xlsx,21.00,CIV,23014,ALGEBRA MODERNA,CIV-PLS23,